In [2]:
# Cell 1: Autoreload setup — picks up changes to src/ files without kernel restart
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../src')

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

print("Setup complete")

Setup complete


## Phase 2 — Feature Engineering

### 1. Building Statistics Consolidation

EDA (Phase 1) found ~20 building/apartment statistic columns clustered in 
`_AVG`/`_MODE`/`_MEDI` triplets, co-missing due to a shared root cause 
(absence of the applicant's building record). This step consolidates each 
triplet to a single `_AVG` column and adds one `BUILDING_INFO_AVAILABLE` flag.

### 0. DAYS_EMPLOYED Fix (moved earlier in pipeline)

Originally explored only in EDA, this must run **before** other Phase 2 steps: 
investigating the OCCUPATION_TYPE missing-category issue (below) revealed 99.96% 
overlap between "not employed" (DAYS_EMPLOYED == 365243) and missing OCCUPATION_TYPE 
— confirming they're largely the same population (pensioners/unemployed with no 
occupation to report). Recreates the EDA fix: IS_NOT_EMPLOYED flag + placeholder 
replaced with NaN.

### 4. Categorical Encoding

... (keep existing target encoding documentation) ...

**Bug found and fixed:** initial target encoding showed 96,391 "unseen categories" 
for OCCUPATION_TYPE when encoding the SAME data it was learned from — impossible 
unless something was wrong. Root cause: pandas groupby() silently drops NaN groups, 
so missing OCCUPATION_TYPE (31.35% of data) never got a learned encoding and fell 
back to a generic global mean. Investigation confirmed missing OCCUPATION_TYPE 
overlaps 99.96% with the not-employed population (IS_NOT_EMPLOYED) — a real, 
informative group, not noise. Fixed by filling NaN with the string 'Missing' 
before grouping in both fit_target_encoding and apply_target_encoding, so it 
receives its own learned, meaningful encoded value.

In [13]:
# Cell 2 (rebuild in correct order)
from feature_engineering import (
    fix_days_employed, consolidate_building_stats, add_ext_source_missing_flags,
    transform_amounts, fit_target_encoding, apply_target_encoding, one_hot_encode_remaining
)

train_raw = pd.read_csv('../data/raw/home-credit-default-risk/application_train.csv')

train_fe = fix_days_employed(train_raw)
train_fe = consolidate_building_stats(train_fe)
train_fe = add_ext_source_missing_flags(train_fe)
train_fe = transform_amounts(train_fe)

encoding_maps = fit_target_encoding(train_fe)
train_fe = apply_target_encoding(train_fe, encoding_maps)
train_fe = one_hot_encode_remaining(train_fe)

print(f"\nFinal shape: {train_fe.shape}")

IS_NOT_EMPLOYED: 55374 flagged (18.01%)
DAYS_EMPLOYED placeholder replaced with NaN
Found 14 building-stat triplets: ['APARTMENTS', 'BASEMENTAREA', 'YEARS_BEGINEXPLUATATION', 'YEARS_BUILD', 'COMMONAREA', 'ELEVATORS', 'ENTRANCES', 'FLOORSMAX', 'FLOORSMIN', 'LANDAREA', 'LIVINGAPARTMENTS', 'LIVINGAREA', 'NONLIVINGAPARTMENTS', 'NONLIVINGAREA']
Dropped 28 redundant MODE/MEDI columns
Kept 14 _AVG columns + 1 new BUILDING_INFO_AVAILABLE flag
EXT_SOURCE_1_MISSING: 173378 flagged (56.38%)
EXT_SOURCE_3_MISSING: 60965 flagged (19.83%)
AMT_INCOME_TOTAL: capped 3 rows at 10,000,000, log-transformed
AMT_CREDIT: log-transformed (no capping - ceiling confirmed legitimate in EDA)
Learned encoding for ORGANIZATION_TYPE: 58 categories (including 'Missing' if present)
Learned encoding for OCCUPATION_TYPE: 19 categories (including 'Missing' if present)
ORGANIZATION_TYPE: encoded, 0 truly-unseen categories filled with global mean (0.0791)
OCCUPATION_TYPE: encoded, 0 truly-unseen categories filled with globa

**Correction:** the initial flag used only `APARTMENTS_AVG` as a reference column, 
which undercounted availability — some applicants had data in other triplets (e.g. 
`YEARS_BEGINEXPLUATATION_AVG`) even when missing `APARTMENTS_AVG`. Fixed to check 
**any** of the 14 `_AVG` columns via `.any(axis=1)`, which more accurately captures 
"does this applicant have any building record at all."

**Corrected result:** 158,701 applicants (51.6%) have at least partial building 
info vs. 148,810 (48.4%) with none — a near-even split, revised from the earlier 
(undercounted) 156,061/151,450 split.

In [4]:
# Cell 3: Sanity-check the new flag
print(train_fe['BUILDING_INFO_AVAILABLE'].value_counts())
print(f"\nRetained columns sample:")
print([c for c in train_fe.columns if 'AVG' in c or 'BUILDING' in c])

BUILDING_INFO_AVAILABLE
1    158701
0    148810
Name: count, dtype: int64

Retained columns sample:
['APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMAX_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG', 'BUILDING_INFO_AVAILABLE']


In [5]:
# Cell 4: Does building info availability correlate with default risk?
check = train_fe.groupby('BUILDING_INFO_AVAILABLE')['TARGET'].agg(['count', 'mean'])
check.columns = ['count', 'default_rate']
check['default_rate_pct'] = check['default_rate'] * 100
print(check)

                          count  default_rate  default_rate_pct
BUILDING_INFO_AVAILABLE                                        
0                        148810      0.092171          9.217123
1                        158701      0.070000          6.999956


**Finding:** Default rate confirms the same pattern with corrected counts: 9.22% 
(no building info) vs. 7.00% (building info available), both deviating from the 
8.07% baseline in the same direction as before. The relationship is stable across 
both the original and corrected flag logic — strengthens confidence this is a real 
signal (likely tied to housing stability/homeownership) rather than a counting 
artifact. Flag retained as a feature for Phase 3.

### 2. EXT_SOURCE Missingness Flags

EDA found that missingness in `EXT_SOURCE_1` and `EXT_SOURCE_3` carries predictive 
signal beyond their raw values (8.52% vs 7.50% default rate for EXT_SOURCE_1; 
9.31% vs 7.77% for EXT_SOURCE_3). Creating binary flags here, before any imputation, 
preserves this signal as a standalone feature. `EXT_SOURCE_2` excluded — only 0.21% 
missing, too rare to be a useful flag.

In [6]:
# Cell 5: Apply EXT_SOURCE missingness flags
from feature_engineering import add_ext_source_missing_flags

train_fe = add_ext_source_missing_flags(train_fe)
print(f"Shape after: {train_fe.shape}")    

EXT_SOURCE_1_MISSING: 173378 flagged (56.38%)
EXT_SOURCE_3_MISSING: 60965 flagged (19.83%)
Shape after: (307511, 97)


**Result:** Flags created matching EDA findings exactly — `EXT_SOURCE_1_MISSING` 
56.38% (173,378), `EXT_SOURCE_3_MISSING` 19.83% (60,965). Shape: 95 → 97 columns.

### 3. Amount Transforms — Income and Credit

EDA found both `AMT_INCOME_TOTAL` and `AMT_CREDIT` heavily right-skewed, with 
log-transform producing a usable distribution for both. EDA also found and 
validated a single genuine outlier in `AMT_INCOME_TOTAL` (117,000,000 — isolated, 
not part of a legitimate heavy tail) which is capped at 10,000,000 before 
log-transforming. `AMT_CREDIT`'s max (4,050,000) was confirmed to be a legitimate 
loan product ceiling — no capping applied there.

In [7]:
# Cell 6: Apply amount transforms
from feature_engineering import transform_amounts

train_fe = transform_amounts(train_fe)
print(f"Shape after: {train_fe.shape}")

print(train_fe[['AMT_INCOME_TOTAL', 'AMT_INCOME_TOTAL_LOG', 'AMT_CREDIT', 'AMT_CREDIT_LOG']].describe())

AMT_INCOME_TOTAL: capped 3 rows at 10,000,000, log-transformed
AMT_CREDIT: log-transformed (no capping - ceiling confirmed legitimate in EDA)
Shape after: (307511, 101)
       AMT_INCOME_TOTAL  AMT_INCOME_TOTAL_LOG    AMT_CREDIT  AMT_CREDIT_LOG
count      3.075110e+05         307511.000000  3.075110e+05   307511.000000
mean       1.684126e+05             11.909234  5.990260e+05       13.070108
std        1.056929e+05              0.488791  4.024908e+05        0.715193
min        2.565000e+04             10.152338  4.500000e+04       10.714440
25%        1.125000e+05             11.630717  2.700000e+05       12.506181
50%        1.471500e+05             11.899215  5.135310e+05       13.149068
75%        2.025000e+05             12.218500  8.086500e+05       13.603123
max        1.000000e+07             16.118096  4.050000e+06       15.214228


**Result:** 3 rows capped in `AMT_INCOME_TOTAL` (matching EDA exactly), both 
columns log-transformed. Raw values preserved as `_RAW` columns for reference. 
Shape: 97 → 101 columns.

**Note:** hit a `NameError: name 'np' is not defined` on first run — `numpy` 
import was missing from the top of `feature_engineering.py` (notebook-level 
imports don't carry into imported modules). Fixed by adding `import numpy as np` 
directly in the file.

In [8]:
# Cell 7: Categorical encoding (target encoding + one-hot)
from feature_engineering import fit_target_encoding, apply_target_encoding, one_hot_encode_remaining

# Step 1: learn the target encoding from train_fe (this IS our training data)
encoding_maps = fit_target_encoding(train_fe)

# Step 2: apply it to train_fe itself
train_fe = apply_target_encoding(train_fe, encoding_maps)

# Step 3: one-hot encode everything else
train_fe = one_hot_encode_remaining(train_fe)

print(f"\nShape after: {train_fe.shape}")

Learned encoding for ORGANIZATION_TYPE: 58 categories
Learned encoding for OCCUPATION_TYPE: 18 categories
ORGANIZATION_TYPE: encoded, 0 unseen categories filled with global mean (0.0791)
OCCUPATION_TYPE: encoded, 96391 unseen categories filled with global mean (0.0863)
One-hot encoding 14 columns: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']

Shape after: (307511, 165)


In [9]:
# Cell 8: Check OCCUPATION_TYPE missingness in the pre-encoding dataframe
print(f"Missing OCCUPATION_TYPE in train_raw: {train_raw['OCCUPATION_TYPE'].isnull().sum()}")
print(f"As percentage: {train_raw['OCCUPATION_TYPE'].isnull().sum() / len(train_raw) * 100:.2f}%")

Missing OCCUPATION_TYPE in train_raw: 96391
As percentage: 31.35%


In [12]:
# Cell 9 (corrected): Check overlap using DAYS_EMPLOYED placeholder directly
occ_missing = train_raw['OCCUPATION_TYPE'].isnull()
not_employed = (train_raw['DAYS_EMPLOYED'] == 365243)

overlap = pd.crosstab(occ_missing, not_employed)
overlap.index = ['Occupation present', 'Occupation missing']
overlap.columns = ['Employed', 'Not employed (365243)']
print(overlap)

                    Employed  Not employed (365243)
Occupation present    211118                      2
Occupation missing     41019                  55372


In [14]:
# Cell 10: Check what the 'Missing' category learned for OCCUPATION_TYPE
print(encoding_maps['OCCUPATION_TYPE'].sort_values())

OCCUPATION_TYPE
Accountants              0.048303
High skill tech staff    0.061599
Managers                 0.062140
Core staff               0.063040
HR staff                 0.063943
IT staff                 0.064639
Missing                  0.065131
Private service staff    0.065988
Medicine staff           0.067002
Secretaries              0.070498
Realty agents            0.078562
Cleaning staff           0.096067
Sales staff              0.096318
Cooking staff            0.104440
Laborers                 0.105788
Security staff           0.107424
Waiters/barmen staff     0.112760
Drivers                  0.113261
Low-skill Laborers       0.171524
Name: TARGET, dtype: float64


**Verification:** the learned 'Missing' encoding for OCCUPATION_TYPE = 0.0651 
(6.51% default rate) — placing it in the lower-risk half of the distribution, 
between IT staff (6.46%) and Private service staff (6.60%), consistent with the 
pensioner-dominated population it represents. This confirms the fix was not 
cosmetic: the previous buggy fallback (generic global mean, 0.0852) would have 
meaningfully mis-encoded this group's true (lower) risk for 31% of the dataset. 

Broader pattern: occupation default rates form a clear socioeconomic gradient — 
white-collar/skilled roles cluster at 5-7% (Accountants, Managers, IT staff), 
manual/physical labor roles cluster at 10-17% (Low-skill Laborers, Drivers, 
Waiters) — consistent with the education and region-tier gradients found in EDA.